# Kodra AI Agent: Cloud/Colab GPU Training Preparation Notebook

**Product:** Kodra AI Agent  
**Model:** Kodra GPT (`KodraGPT`)  
**Core:** Kodra Core  
**Tagline:** CODE • THINK • CREATE

This notebook prepares and validates a GPU training run for **Kodra GPT** on Google Colab (or any Jupyter environment with a CUDA GPU). It clones the repository, prepares an approved dataset, trains or loads a BPE tokenizer, selects a Kodra Tiny/Small configuration with its exact parameter count, runs a 20-step GPU smoke test with train/validation loss reporting, saves and reloads a checkpoint, resumes for 5 more steps, and optionally backs checkpoints up to Google Drive.

Full multi-epoch training is behind an explicit **RUN MANUALLY ONLY** gate near the end of the notebook — it will not run as part of "Run All".

No credentials are embedded in this notebook. If you want to persist checkpoints to Google Drive, mount it yourself in Colab and pass that path as `CHECKPOINT_DIR` below.

In [4]:
#!/bin/bash
# 1. Clone the repository and enter the Kodra Core directory
!rm -rf /content/Kodra-ai
!git clone https://github.com/ChaceEthan/Kodra-ai.git /content/Kodra-ai
%cd /content/Kodra-ai/kodra-core

Cloning into '/content/Kodra-ai'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 169 (delta 45), reused 164 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 148.98 KiB | 1.23 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/Kodra-ai/kodra-core


In [6]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [7]:
# 3. Detect GPU and print info
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))
    print('Device count:', torch.cuda.device_count())
    print('Total memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('No GPU detected - training will fall back to CPU (slow for anything above kodra-tiny).')

CUDA available: True
Device name: Tesla T4
Device count: 1
Total memory (GB): 15.64


In [24]:
# 4. (Optional) Mount Google Drive for persistent checkpoint storage.
# Uncomment if you want checkpoints to survive a Colab session restart.
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/kodra_checkpoints'
CHECKPOINT_DIR = 'checkpoints'

In [9]:
# 5. Prepare dataset: build a manifest from an approved/licensed source tree.
# Point SOURCE_DIR at a directory you have the rights to train on. The bundled
# `data/code` sample corpus is used by default so this cell always runs.
from datasets.corpus_pipeline import build_manifest, write_manifest

SOURCE_DIR = 'data/code'
manifest = build_manifest(SOURCE_DIR, seed=42, license='project-sample', source='kodra-sample-corpus')
write_manifest(manifest, 'data/manifest.json')
print(f'Discovered {manifest.num_files} files, {manifest.total_chars} chars, languages: {manifest.language_counts}')

Discovered 4 files, 733 chars, languages: {'python': 1, 'json': 1, 'markdown': 1, 'typescript': 1}


In [10]:
# 6. Train (or load) the tokenizer. BPE is the default for GPU runs; the
# Phase 1 char tokenizer remains available for parity/debug comparisons.
import os
from datasets.sample_code import SAMPLE_CODE_CORPUS
from tokenizer.char_tokenizer import CharTokenizer
from tokenizer.bpe_tokenizer import ByteLevelBPETokenizer

USE_BPE = True  # set False to use the Phase 1 char tokenizer instead
BPE_VOCAB_PATH = 'tokenizer/vocab_bpe.json'
CHAR_VOCAB_PATH = 'tokenizer/vocab.json'

if USE_BPE:
    tokenizer = ByteLevelBPETokenizer(vocab_size=8000)
    if os.path.exists(BPE_VOCAB_PATH):
        tokenizer.load(BPE_VOCAB_PATH)
        print(f'Loaded existing BPE tokenizer from {BPE_VOCAB_PATH}')
    else:
        tokenizer.train(SAMPLE_CODE_CORPUS)
        tokenizer.save(BPE_VOCAB_PATH)
        print(f'Trained a new BPE tokenizer and saved it to {BPE_VOCAB_PATH}')
else:
    tokenizer = CharTokenizer()
    if os.path.exists(CHAR_VOCAB_PATH):
        tokenizer.load(CHAR_VOCAB_PATH)
        print(f'Loaded existing char tokenizer from {CHAR_VOCAB_PATH}')
    else:
        tokenizer.train(SAMPLE_CODE_CORPUS)
        tokenizer.save(CHAR_VOCAB_PATH)
        print(f'Trained a new char tokenizer and saved it to {CHAR_VOCAB_PATH}')

print('Tokenizer type:', tokenizer.tokenizer_type, '| vocab size:', tokenizer.vocab_size)

Trained a new BPE tokenizer and saved it to tokenizer/vocab_bpe.json
Tokenizer type: bpe | vocab size: 415


In [11]:
# 7. Select a Kodra model configuration
from configs.model_sizes import get_model_size, validate_model_config, estimate_resources
from model.gpt_model import KodraGPT

MODEL_SIZE = 'kodra-tiny'  # one of: kodra-tiny, kodra-small (kodra-base/kodra-medium are roadmap-only)
spec = get_model_size(MODEL_SIZE)
model_cfg = spec.config
model_cfg.vocab_size = tokenizer.vocab_size
validate_model_config(model_cfg)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KodraGPT(model_cfg).to(device)

# Exact parameter count comes from the instantiated model, not the roadmap estimate.
param_count = model.count_parameters()
resource_estimate = estimate_resources(model_cfg)
print(f'{spec.display_name}: {param_count:,} exact parameters ({param_count/1e6:.2f}M) on {device}')
print(f'Previously trained in this repo: {spec.trained}')
print(f'Rough planning estimate: {resource_estimate["parameters"]:,} params | '
      f'training_vram~{resource_estimate["training_vram_gb"]:.2f}GB | '
      f'inference_vram~{resource_estimate["inference_vram_gb"]:.2f}GB')

Kodra Tiny: 4,910,848 exact parameters (4.91M) on cuda
Previously trained in this repo: True
Rough planning estimate: 4,910,848 params | training_vram~0.07GB | inference_vram~0.01GB


In [13]:
# 8. Build train/validation dataloaders and the trainer
from configs.config import TrainingConfig
from datasets.dataset import create_dataloader
from training.trainer import Trainer
from training.utils import set_seed

set_seed(42)

# Simple held-out split of the sample corpus for a train/validation loss signal.
_split_idx = int(len(SAMPLE_CODE_CORPUS) * 0.9)
TRAIN_TEXT = SAMPLE_CODE_CORPUS[:_split_idx]
VAL_TEXT = SAMPLE_CODE_CORPUS[_split_idx:]

train_cfg = TrainingConfig(batch_size=16, learning_rate=3e-4, max_epochs=10)
train_loader = create_dataloader(TRAIN_TEXT, tokenizer, model_cfg.context_length, train_cfg.batch_size)
val_loader = create_dataloader(VAL_TEXT, tokenizer, model_cfg.context_length, train_cfg.batch_size, shuffle=False)

dataset_manifest_id = f"{manifest.source}-seed{manifest.seed}-{manifest.created_at}"

trainer = Trainer(
    model, train_cfg, train_loader, val_loader=val_loader, device=device,
    tokenizer_type=tokenizer.tokenizer_type, dataset_manifest_id=dataset_manifest_id,
)

In [14]:
# 9. GPU smoke training: run a fixed 20 optimizer steps to validate the
# pipeline end-to-end (forward/backward/step, AMP, checkpointing) before
# committing to a full run. This is NOT the full training loop.
SMOKE_TRAIN_STEPS = 20

smoke_epoch = 0
while trainer.step_count < SMOKE_TRAIN_STEPS:
    smoke_epoch += 1
    smoke_avg_loss = trainer.train_epoch(smoke_epoch, total_steps=SMOKE_TRAIN_STEPS)
    print(f'[smoke] epoch {smoke_epoch} | avg_loss={smoke_avg_loss:.4f} | step={trainer.step_count}')

print(f'Smoke training complete at step {trainer.step_count} (target was {SMOKE_TRAIN_STEPS}).')

[smoke] epoch 1 | avg_loss=5.9333 | step=1
[smoke] epoch 2 | avg_loss=5.9055 | step=2
[smoke] epoch 3 | avg_loss=5.8167 | step=3
[smoke] epoch 4 | avg_loss=5.6957 | step=4
[smoke] epoch 5 | avg_loss=5.6025 | step=5
[smoke] epoch 6 | avg_loss=5.4551 | step=6
[smoke] epoch 7 | avg_loss=5.2887 | step=7
[smoke] epoch 8 | avg_loss=5.1509 | step=8
[smoke] epoch 9 | avg_loss=5.0253 | step=9
[smoke] epoch 10 | avg_loss=4.9128 | step=10
[smoke] epoch 11 | avg_loss=4.8173 | step=11
[smoke] epoch 12 | avg_loss=4.7150 | step=12
[smoke] epoch 13 | avg_loss=4.6355 | step=13
[smoke] epoch 14 | avg_loss=4.5552 | step=14
[smoke] epoch 15 | avg_loss=4.4733 | step=15
[smoke] epoch 16 | avg_loss=4.4110 | step=16
[smoke] epoch 17 | avg_loss=4.3335 | step=17
[smoke] epoch 18 | avg_loss=4.2726 | step=18
[smoke] epoch 19 | avg_loss=4.2170 | step=19
[smoke] epoch 20 | avg_loss=4.1615 | step=20
Smoke training complete at step 20 (target was 20).


### Extended Training for Syntactic Improvement
We will now run a more substantial training session (50 epochs) to help the model internalize Python syntax rules.

In [36]:
# Update configuration for longer training
extended_train_cfg = TrainingConfig(batch_size=16, learning_rate=3e-4, max_epochs=50)

# Re-initialize trainer with extended config
extended_trainer = Trainer(
    model, extended_train_cfg, train_loader, val_loader=val_loader, device=device,
    tokenizer_type=tokenizer.tokenizer_type, dataset_manifest_id=dataset_manifest_id,
)

# Run extended training loop
print("Starting extended training loop...")
total_steps = extended_train_cfg.max_epochs * len(train_loader)
for epoch in range(1, extended_train_cfg.max_epochs + 1):
    avg_loss = extended_trainer.train_epoch(epoch, total_steps=total_steps)
    if epoch % 10 == 0:
        val_loss = extended_trainer.evaluate()
        print(f'Epoch {epoch}/{extended_train_cfg.max_epochs} | Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f}')

print("Extended training complete. Re-running evaluation...")

Starting extended training loop...
Epoch 10/50 | Loss: 3.0199 | Val Loss: 1.3725
Epoch 20/50 | Loss: 2.7323 | Val Loss: 1.2701
Epoch 30/50 | Loss: 2.3111 | Val Loss: 1.1225
Epoch 40/50 | Loss: 1.8592 | Val Loss: 0.9264
Epoch 50/50 | Loss: 1.4293 | Val Loss: 0.7418
Extended training complete. Re-running evaluation...


In [ ]:
# Re-evaluate syntax after extended training
new_report = full_evaluation_report(model, tokenizer, device, train_loader)
print(f"New Python Parse Rate: {new_report['syntax']['python_parse_rate'] * 100}%")

# Show sample generation
generator = CodeGenerator(model, tokenizer, device)
print("\nSample Completion:")
print(generator.generate('def quicksort(arr):', max_new_tokens=64))

In [41]:
import os

print("--- Final Verification & Checkpoint Persistence ---")
print(f"Current device: {device}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

# Check if the extended trainer exists from the previous run
if 'extended_trainer' in globals():
    print(f"Original trainer steps (smoke/resume): {trainer.step_count}")
    print(f"Extended trainer steps (post-50 epochs): {extended_trainer.step_count}")

    # Verify final validation loss
    extended_val_loss = extended_trainer.evaluate()
    print(f"Extended model final validation loss: {extended_val_loss:.4f}")

    # Save the updated weights as the latest/best
    extended_trainer.save_latest_and_best(
        CHECKPOINT_DIR,
        val_loss=extended_val_loss
    )

    print("\n✅ EXTENDED MODEL CHECKPOINT SAVED")
    print(f"Path: {os.path.join(CHECKPOINT_DIR, 'kodra_gpt_latest.pt')}")
else:
    print("❌ extended_trainer not found. Please ensure the extended training cell was executed.")

--- Final Verification & Checkpoint Persistence ---
Current device: cuda
Checkpoint directory: checkpoints
Original trainer steps (smoke/resume): 32
Extended trainer steps (post-50 epochs): 50
Extended model final validation loss: 0.7418

✅ EXTENDED MODEL CHECKPOINT SAVED
Path: checkpoints/kodra_gpt_latest.pt


In [25]:
# 10. Train, checkpointing periodically (every epoch here; adjust to your needs)
total_steps = train_cfg.max_epochs * len(train_loader)
for epoch in range(1, train_cfg.max_epochs + 1):
    avg_loss = trainer.train_epoch(epoch, total_steps=total_steps)
    val_loss = trainer.evaluate()  # None unless a val_loader was configured
    trainer.save_latest_and_best(CHECKPOINT_DIR, val_loss=val_loss)
    tps = trainer.history[-1]['tokens_per_sec'] if trainer.history else 0.0
    print(f'Epoch {epoch}/{train_cfg.max_epochs} | avg_loss={avg_loss:.4f} | step={trainer.step_count} | tokens/sec={tps:.0f}')

Epoch 1/10 | avg_loss=3.9962 | step=23 | tokens/sec=4
Epoch 2/10 | avg_loss=3.9409 | step=24 | tokens/sec=4
Epoch 3/10 | avg_loss=3.8753 | step=25 | tokens/sec=4
Epoch 4/10 | avg_loss=3.8261 | step=26 | tokens/sec=5
Epoch 5/10 | avg_loss=3.7567 | step=27 | tokens/sec=5
Epoch 6/10 | avg_loss=3.6922 | step=28 | tokens/sec=5
Epoch 7/10 | avg_loss=3.6415 | step=29 | tokens/sec=5
Epoch 8/10 | avg_loss=3.5619 | step=30 | tokens/sec=5
Epoch 9/10 | avg_loss=3.5395 | step=31 | tokens/sec=5
Epoch 10/10 | avg_loss=3.4346 | step=32 | tokens/sec=6


In [28]:
print(f'CHECKPOINT_DIR is set to: {CHECKPOINT_DIR}')

CHECKPOINT_DIR is set to: checkpoints


In [29]:
# 11. Evaluate (language-model + code-completion + syntax)
from evaluation.evaluator import full_evaluation_report
import json as _json

report = full_evaluation_report(model, tokenizer, device, train_loader)
print(_json.dumps(report, indent=2, default=str))

{
  "language_model": {
    "val_loss": 3.2134740352630615,
    "perplexity": 24.865319345641673
  },
  "code_completion": {
    "total": 7,
    "passed": 7,
    "pass_rate": 1.0,
    "results": [
      {
        "id": "py_function_def",
        "category": "python",
        "language": "python",
        "prompt": "def add(a, b):\n    return",
        "completion": "def add(a, b):\n    return= ):\n     + = = ",
        "passed": true
      },
      {
        "id": "py_loop",
        "category": "python",
        "language": "python",
        "prompt": "for i in range(10):\n   ",
        "completion": "for i in range(10):\n   righ(arr",
        "passed": true
      },
      {
        "id": "js_function_def",
        "category": "javascript",
        "language": "javascript",
        "prompt": "function add(a, b) {\n  return",
        "completion": "function add(a, b) {\n  return arrhKodraNode\ufffd:\n        ",
        "passed": true
      },
      {
        "id": "ts_interface",
      

In [33]:
# 12. Generate a sample code completion with the trained model
from inference.generator import CodeGenerator

generator = CodeGenerator(model, tokenizer, device)
sample = generator.generate('def quicksort(arr):', max_new_tokens=64, temperature=0.7, top_k=40)
print(sample)

def quicksort(arr):(arr = [x for x in arr if x <�


In [34]:
# 13. 5-step resume: continue training the reloaded trainer for 5 more steps

# FIX: Define reload_trainer by loading a checkpoint
import os
from training.trainer import Trainer

# Create a new Trainer instance, reusing variables from previous setup cells.
# These variables (model, train_cfg, train_loader, val_loader, device, tokenizer.tokenizer_type, dataset_manifest_id)
# are expected to be available from cells P4GhsAQ825A1 and earlier.
reload_trainer = Trainer(
    model, train_cfg, train_loader, val_loader=val_loader, device=device,
    tokenizer_type=tokenizer.tokenizer_type, dataset_manifest_id=dataset_manifest_id,
)

# Load the latest checkpoint from the CHECKPOINT_DIR
latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'kodra_gpt_latest.pt')
if os.path.exists(latest_checkpoint_path):
    reload_trainer.load_checkpoint(latest_checkpoint_path)
    print(f'Reloaded trainer from checkpoint: {latest_checkpoint_path}')
else:
    print(f'Warning: No checkpoint found at {latest_checkpoint_path}. ')
    print('reload_trainer will start from scratch, which might not be intended for resume.')

RESUME_STEPS = 5
resume_target = reload_trainer.step_count + RESUME_STEPS
resume_epoch = 0
while reload_trainer.step_count < resume_target:
    resume_epoch += 1
    resume_avg_loss = reload_trainer.train_epoch(resume_epoch, total_steps=resume_target)
    print(f'[resume] epoch {resume_epoch} | avg_loss={resume_avg_loss:.4f} | step={reload_trainer.step_count}')

print(f'Resumed training to step {reload_trainer.step_count} (target was {resume_target}).')


Reloaded trainer from checkpoint: checkpoints/kodra_gpt_latest.pt
[resume] epoch 1 | avg_loss=3.3741 | step=33
[resume] epoch 2 | avg_loss=3.3389 | step=34
[resume] epoch 3 | avg_loss=3.2492 | step=35
[resume] epoch 4 | avg_loss=3.2182 | step=36
[resume] epoch 5 | avg_loss=3.2075 | step=37
Resumed training to step 37 (target was 37).


In [38]:
# 14. (Optional) Back up checkpoints to Google Drive. Only runs if Drive is
# mounted (see the optional cell above) and BACKUP_TO_DRIVE is set True.
BACKUP_TO_DRIVE = False
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/kodra_checkpoints_backup'

if BACKUP_TO_DRIVE:
    import shutil
    if not os.path.isdir('/content/drive'):
        print('Google Drive is not mounted - skipping backup. Mount it in the cell above first.')
    else:
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        for fname in os.listdir(CHECKPOINT_DIR):
            shutil.copy2(os.path.join(CHECKPOINT_DIR, fname), os.path.join(DRIVE_BACKUP_DIR, fname))
        print(f'Backed up checkpoints from {CHECKPOINT_DIR} to {DRIVE_BACKUP_DIR}')
else:
    print('BACKUP_TO_DRIVE is False - skipping Drive backup.')

BACKUP_TO_DRIVE is False - skipping Drive backup.


In [46]:
# 15. FULL TRAINING — RUN MANUALLY ONLY
#
# The smoke test above already validated the pipeline. Full training is a
# long-running, resource-consuming operation and must be started deliberately.
# Flip RUN_FULL_TRAINING to True yourself to proceed; it defaults to False so
# re-running the whole notebook (e.g. "Run All") never triggers a full run.
RUN_FULL_TRAINING = False

if not RUN_FULL_TRAINING:
    raise RuntimeError(
        'Full training is gated. Set RUN_FULL_TRAINING = True above and re-run this '
        'cell to start the full training loop.'
    )

# Resume from the latest checkpoint if one exists (continues past the smoke/resume steps above).
full_ckpt = os.path.join(CHECKPOINT_DIR, 'kodra_gpt_latest.pt')
if os.path.exists(full_ckpt):
    trainer.load_checkpoint(full_ckpt)
    print(f'Resumed full training from step {trainer.step_count}')
else:
    print('No existing checkpoint found - starting full training fresh.')

total_steps = train_cfg.max_epochs * len(train_loader)
for epoch in range(1, train_cfg.max_epochs + 1):
    avg_loss = trainer.train_epoch(epoch, total_steps=total_steps)
    val_loss = trainer.evaluate()
    trainer.save_latest_and_best(CHECKPOINT_DIR, val_loss=val_loss)
    tps = trainer.history[-1]['tokens_per_sec'] if trainer.history else 0.0
    print(f'Epoch {epoch}/{train_cfg.max_epochs} | avg_loss={avg_loss:.4f} | '
          f'val_loss={val_loss} | step={trainer.step_count} | tokens/sec={tps:.0f}')

RuntimeError: Full training is gated. Set RUN_FULL_TRAINING = True above and re-run this cell to start the full training loop.

In [42]:
# 16. Evaluate (language-model + code-completion + syntax) - Post Extended Training
from evaluation.evaluator import full_evaluation_report
import json as _json

# We use the existing model and tokenizer which are now updated from extended training
new_report = full_evaluation_report(model, tokenizer, device, train_loader)

print("--- Updated Evaluation Report ---")
print(f"New Python Parse Rate: {new_report['syntax']['python_parse_rate'] * 100}%")
print(_json.dumps(new_report, indent=2, default=str))

--- Updated Evaluation Report ---
New Python Parse Rate: 0.0%
{
  "language_model": {
    "val_loss": 1.1273430585861206,
    "perplexity": 3.087442439137943
  },
  "code_completion": {
    "total": 7,
    "passed": 7,
    "pass_rate": 1.0,
    "results": [
      {
        "id": "py_function_def",
        "category": "python",
        "language": "python",
        "prompt": "def add(a, b):\n    return",
        "completion": "def add(a, b):\n    return:\n        \ufffdreturn arr\n    pivot = arr[len(arr) // 2return t = [x for x in arr if x <arlen(arr) middle =  arr\n\npivot]\n    == y",
        "passed": true
      },
      {
        "id": "py_loop",
        "category": "python",
        "language": "python",
        "prompt": "for i in range(10):\n   ",
        "completion": "for i in range(10):\n   , value):\n        pivot]\n    self.head = (arr(arr(arr=  = mid",
        "passed": true
      },
      {
        "id": "js_function_def",
        "category": "javascript",
        "langua

In [43]:
# 17. Generate a sample code completion with the improved model
from inference.generator import CodeGenerator

generator = CodeGenerator(model, tokenizer, device)
print("Sample Completion (quicksort):")
sample = generator.generate('def quicksort(arr):', max_new_tokens=64, temperature=0.7, top_k=40)
print(sample)

Sample Completion (quicksort):
def quicksort(arr):target:
            low) 


In [44]:
# Final debug: Inspect token IDs for punctuation to see if they are in vocabulary
punctuation_checks = [':', '    ', '\n', '(', ')']
print("--- Tokenizer Punctuation Check ---")
for p in punctuation_checks:
    tokens = tokenizer.encode(p)
    print(f"Sequence: {repr(p)} | Token IDs: {tokens} | Decoded: {repr(tokenizer.decode(tokens))}")

--- Tokenizer Punctuation Check ---
Sequence: ':' | Token IDs: [62] | Decoded: ':'
Sequence: '    ' | Token IDs: [261] | Decoded: '    '
Sequence: '\n' | Token IDs: [14] | Decoded: '\n'
Sequence: '(' | Token IDs: [44] | Decoded: '('
Sequence: ')' | Token IDs: [45] | Decoded: ')'


In [47]:
from evaluation.evaluator import full_evaluation_report
import json

# Targeting the model from the extended training session
trained_model = extended_trainer.model

print("--- Final Evaluation of Extended Model ---")
report_after_extended_training = full_evaluation_report(
    trained_model,
    tokenizer,
    device,
    train_loader
)

print(f"Final Python Parse Rate: {report_after_extended_training['syntax']['python_parse_rate'] * 100}%")
print(json.dumps(report_after_extended_training, indent=2, default=str))

--- Final Evaluation of Extended Model ---
Final Python Parse Rate: 0.0%
{
  "language_model": {
    "val_loss": 1.1273430585861206,
    "perplexity": 3.087442439137943
  },
  "code_completion": {
    "total": 7,
    "passed": 7,
    "pass_rate": 1.0,
    "results": [
      {
        "id": "py_function_def",
        "category": "python",
        "language": "python",
        "prompt": "def add(a, b):\n    return",
        "completion": "def add(a, b):\n    return 1",
        "passed": true
      },
      {
        "id": "py_loop",
        "category": "python",
        "language": "python",
        "prompt": "for i in range(10):\n   ",
        "completion": "for i in range(10):\n   lef",
        "passed": true
      },
      {
        "id": "js_function_def",
        "category": "javascript",
        "language": "javascript",
        "prompt": "function add(a, b) {\n  return",
        "completion": "function add(a, b) {\n  returnypivot]\n    <h(self\n        if arr[mid] = ",
        "pa

### Diagnostic 1: Data Serialization Inspection
Checking if the training data contains accidental metadata (JSON-like fields) that the model is learning.

In [48]:
import json
import os

# Inspect the manifest used for training
manifest_path = 'data/manifest.json'
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest_data = json.load(f)
    print("--- Manifest Metadata ---")
    print(f"Source: {manifest_data.get('source')}")

# Inspect sample of actual text provided to the trainer
print("\n--- Raw Training Text Sample (First 500 chars) ---")
print(repr(TRAIN_TEXT[:500]))

# Check if labels like 'target:' or 'prompt:' appear in the corpus
for indicator in ['target:', 'prompt:', 'completion:', 'category:']:
    if indicator in TRAIN_TEXT:
        print(f"CRITICAL: Metadata tag '{indicator}' found in training text.")

--- Manifest Metadata ---
Source: kodra-sample-corpus

--- Raw Training Text Sample (First 500 chars) ---
'\ndef kodra_ai_quick_sort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[len(arr) // 2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return kodra_ai_quick_sort(left) + middle + kodra_ai_quick_sort(right)\n\ndef binary_search(arr, target):\n    low = 0\n    high = len(arr) - 1\n    while low <= high:\n        mid = (low + high) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr['
CRITICAL: Metadata tag 'target:' found in training text.


### Diagnostic 2: Tokenizer Round-Trip & Special Tokens
Verifying if the tokenizer handles multi-line Python code correctly and checking special token IDs.

In [49]:
python_sample = """def test_func(x):
    return x + 1"""
encoded = tokenizer.encode(python_sample)
decoded = tokenizer.decode(encoded)

print(f"Original:\n{repr(python_sample)}")
print(f"Decoded:\n{repr(decoded)}")
print(f"Match: {python_sample == decoded}")

# Check for special tokens in the BPE tokenizer
try:
    print(f"\nBOS token: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
    print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
except AttributeError:
    print("Tokenizer does not have explicit BOS/EOS attributes.")

Original:
'def test_func(x):\n    return x + 1'
Decoded:
'def test_func(x):\n    return x + 1'
Match: True
Tokenizer does not have explicit BOS/EOS attributes.


### Diagnostic 3: Evaluation & Generation Logic
Checking `evaluation/evaluator.py` and `inference/generator.py` to see how prompts are handled.

In [52]:
import inspect
from evaluation.evaluator import run_syntax_eval

# Check how syntax evaluation is performed
print("--- run_syntax_eval Source ---")
try:
    print(inspect.getsource(run_syntax_eval))
except Exception as e:
    print(f"Could not get source: {e}")

# Verify if the sample corpus contains metadata-heavy files
print("\n--- Files in data/code ---")
!ls -R data/code

# Check for mixed content in the Python sample
print("\n--- Python Data Content Check ---")
!cat data/code/sample.py | head -n 20

--- run_syntax_eval Source ---
def run_syntax_eval(generator: CodeGenerator, num_python_samples: int = 5, max_new_tokens: int = 48) -> Dict[str, Any]:
    prompts = ["def ", "class ", "import ", "for i in range(10):\n", "if x > 0:\n"][:num_python_samples]
    python_results = []
    for p in prompts:
        code = generator.generate(p, max_new_tokens=max_new_tokens, temperature=0.7, top_k=40)
        python_results.append({"prompt": p, "code": code, "parses": python_parses(code)})

    parses_count = sum(1 for r in python_results if r["parses"])
    return {
        "python_parse_rate": (parses_count / len(python_results)) if python_results else 0.0,
        "python_results": python_results,
    }


--- Files in data/code ---
data/code:
algorithms.py  config.json  doc.md  types.ts

--- Python Data Content Check ---
cat: data/code/sample.py: No such file or directory


In [53]:
import torch
from training.utils import set_seed

# CELL 18 — DATASET & DATALOADER INSPECTION
print("--- Inspecting Training Samples ---")
set_seed(42)

# Fetch a single batch from the loader used during training
batch_iter = iter(train_loader)
inputs, targets = next(batch_iter)

# Inspect the first 5 samples in the batch
for i in range(min(5, len(inputs))):
    input_ids = inputs[i].tolist()
    target_ids = targets[i].tolist()

    decoded_input = tokenizer.decode(input_ids)
    decoded_target = tokenizer.decode(target_ids)

    print(f"\nSample {i+1}:")
    print(f"Decoded Input (Tokens 0-10): {repr(tokenizer.decode(input_ids[:10]))}...")
    print(f"Decoded Target (Tokens 0-10): {repr(tokenizer.decode(target_ids[:10]))}...")
    print(f"Full Decoded Input Content:\n{repr(decoded_input)}")

# Specific check for metadata leakage in the actual Tensors
metadata_keys = ['target:', 'prompt:', 'completion:', '{', '"']
for key in metadata_keys:
    if key in decoded_input:
        print(f"\n[!] Metadata key '{key}' found in the decoded training batch input.")

--- Inspecting Training Samples ---

Sample 1:
Decoded Input (Tokens 0-10): '\ndef kodra_ai_quick_sort(arr):\n    if len(arr) <= 1'...
Decoded Target (Tokens 0-10): 'def kodra_ai_quick_sort(arr):\n    if len(arr) <= 1:\n        '...
Full Decoded Input Content:
'\ndef kodra_ai_quick_sort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[len(arr) // 2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return kodra_ai_quick_sort(left) + middle + kodra_ai_quick_sort(right)\n\ndef binary_search(arr, target):\n    low = 0\n    high = len(arr) - 1\n    while low <= high:\n        mid = (low + high) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            low = mid + 1\n        else:\n            high = mid - 1\n    return -1\n\nclass KodraNode:\n    def __init__(self, value):\n        self.value = value\n        self.next = None\n\nclass KodraL

In [54]:
import inspect
from training.trainer import Trainer

# CELL 19 — CAUSAL SHIFT VERIFICATION
print("--- Checking Trainer Causal Shift Logic ---")

try:
    # Inspect the train_step or similar method to see how loss is calculated
    trainer_source = inspect.getsource(Trainer.train_epoch)
    print("Trainer.train_epoch snippet:")
    # Printing only the first part of the loop logic to check shift
    print("\n".join(trainer_source.splitlines()[:30]))
except Exception as e:
    print(f"Could not inspect Trainer source: {e}")

print("\n--- Checking Data/Manifest Source Files ---")
!grep -r "target:" data/code
!grep -r "prompt:" data/code

--- Checking Trainer Causal Shift Logic ---
Trainer.train_epoch snippet:
    def train_epoch(self, epoch: int, total_steps: Optional[int] = None) -> float:
        self.model.train()
        total_loss = 0.0
        num_batches = len(self.train_loader)
        if total_steps:
            self._total_steps_estimate = total_steps

        accum_steps = max(1, self.config.grad_accum_steps)
        self.optimizer.zero_grad(set_to_none=True)

        for batch_idx, (x, y) in enumerate(self.train_loader):
            x, y = x.to(self.device), y.to(self.device)

            with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.use_amp):
                logits, loss = self.model(x, y)
                loss_to_backward = loss / accum_steps

            if torch.isnan(loss).any() or torch.isinf(loss).any():
                self.nan_events += 1
                self.optimizer.zero_grad(set_to_none=True)
                continue

            self.scaler.scale(loss_to_ba

In [55]:
import ast
from inference.generator import CodeGenerator

# CELL 20 — GENERATOR & EVALUATOR INTERACTION TEST
generator = CodeGenerator(model, tokenizer, device)
test_prompts = ["def ", "class ", "import ", "def quicksort(arr):"]

print("--- Generation & AST Parse Trace ---")
for p in test_prompts:
    # 1. Generate
    raw_gen = generator.generate(p, max_new_tokens=48, temperature=0.7)

    # 2. Determine if it includes the prompt
    has_prompt = raw_gen.startswith(p)

    # 3. Simulate the evaluator's ast.parse check
    # We need to see exactly what string is being evaluated
    try:
        ast.parse(raw_gen)
        parse_result = "SUCCESS"
        parse_err = ""
    except Exception as e:
        parse_result = "FAILED"
        parse_err = str(e)

    print(f"\nPrompt: {repr(p)}")
    print(f"Raw Generated Output: {repr(raw_gen)}")
    print(f"Generated contains prompt? {has_prompt}")
    print(f"AST Parse Status: {parse_result}")
    if parse_err:
        print(f"AST Error: {parse_err}")

--- Generation & AST Parse Trace ---

Prompt: 'def '
Raw Generated Output: 'def , '
Generated contains prompt? True
AST Parse Status: FAILED
AST Error: invalid syntax (<unknown>, line 1)

Prompt: 'class '
Raw Generated Output: 'class '
Generated contains prompt? True
AST Parse Status: FAILED
AST Error: invalid syntax (<unknown>, line 1)

Prompt: 'import '
Raw Generated Output: 'import len(arr) �<def 1returndef y=  = :\n            pivot = �lefreturn:\n    def __init__(selfpivot]\n     < return(arr middle + :\n    def __init__(self:\n    def __init__(selfreturn kodra_ai_quick_sort(left)  < = ]\n    Lt)\n\ndef bhigh:\n            ar)len(arr) '
Generated contains prompt? True
AST Parse Status: FAILED
AST Error: invalid character '�' (U+FFFD) (<unknown>, line 1)

Prompt: 'def quicksort(arr):'
Raw Generated Output: 'def quicksort(arr):len(arr) returntarget:\n            return arr\n      = - 1\n    �// 2�lef�< pivot]\n    middle = lef==  pivot]\n    high]\n    Noderighreturn kodra_ai_quick_

In [56]:
# CELL 21 — ROOT CAUSE REPORT

# Placeholder to be filled by the agent after execution results are known.
print("\n--- KODRA AI ROOT-CAUSE REPORT ---")
print("ROOT CAUSE: [Pending execution results]")
print("SECONDARY CAUSES: [Pending execution results]")
print("TOKENIZER: [Check Result]")
print("DATASET FORMAT: [Check Result]")
print("CAUSAL TARGET SHIFT: [Check Result]")
print("GENERATOR: [Check Result]")
print("EVALUATOR: [Check Result]")
print("\nWHY 'target:' APPEARS: [Pending execution results]")
print("FILES RESPONSIBLE: [Pending execution results]")
print("MINIMAL FIX: [Pending execution results]")
print("RETRAIN REQUIRED: [Yes/No]")
print("CHECKPOINT REUSABLE: [Yes/No/Partially]")


--- KODRA AI ROOT-CAUSE REPORT ---
ROOT CAUSE: [Pending execution results]
SECONDARY CAUSES: [Pending execution results]
TOKENIZER: [Check Result]
DATASET FORMAT: [Check Result]
CAUSAL TARGET SHIFT: [Check Result]
GENERATOR: [Check Result]
EVALUATOR: [Check Result]

WHY 'target:' APPEARS: [Pending execution results]
FILES RESPONSIBLE: [Pending execution results]
MINIMAL FIX: [Pending execution results]
RETRAIN REQUIRED: [Yes/No]
CHECKPOINT REUSABLE: [Yes/No/Partially]
